# Study 906 — EM Local Bonds FX-Hedged — the teardown

The EMLC~UUP hedge regression, the excess-vs-excess race, HAC *t*'s, the circular-block bootstrap Sharpe CIs, the walk-forward hedge and its residual EM-FX beta, the era split, drawdowns, the costed overlay, and the planted-carry synthetic control.

In [1]:
R = {'start': '2010-08-31', 'end': '2026-06-30', 'n': 191, 'fingerprint': '32605dae0c9c', 'emlc_uup_beta': -1.12, 't_emlc_uup': -11.91, 'emlc_uup_r2': 0.5, 'hedge_b': -1.118, 'unhedged_exc': 0.33, 'unhedged_sharpe': 0.03, 't_unhedged': 0.14, 'hedged_exc': 1.62, 'hedged_sharpe': 0.2, 't_hedged': 0.94, 'emb_exc': 3.1, 'emb_sharpe': 0.34, 't_emb': 1.48, 'prem_diff': -1.48, 't_prem_diff': -0.81, 'welch': -0.49, 'lemb_sharpe': 0.23, 'lemb_t': 0.99, 'lemb_prem': -1.03, 'ebnd_sharpe': 0.26, 'ebnd_t': 1.16, 'ebnd_prem': -1.33, 'boot_unh_lo': -0.38, 'boot_unh_hi': 0.47, 'boot_hed_lo': -0.21, 'boot_hed_hi': 0.69, 'boot_hed_fracneg': 0.17, 'boot_emb_lo': -0.05, 'boot_emb_hi': 0.91, 'wf_exc': 1.6, 'wf_sharpe': 0.2, 'wf_t': 0.79, 'wf_resid_fx': 0.1, 'era_early_hed': 1.76, 'era_early_t': 0.7, 'era_early_prem': -3.58, 'era_early_prem_t': -1.87, 'era_early_n': 125, 'era_late_hed': 1.34, 'era_late_t': 0.84, 'era_late_prem': 2.5, 'era_late_prem_t': 0.71, 'era_late_n': 66, 'dd_emlc': -32.3, 'dd_emb': -28.7, 'dd_hedged': -17.1, 'cost_charge': 0.46, 'cost_gross': 1.62, 'cost_net': 1.15, 'cost_net_t': 0.67, 'cost_net_sharpe': 0.15, 'cost_net_prem': -1.95, 'cost_net_prem_t': -1.07, 'planted_exc': 5.53, 'planted_t': 3.25, 'planted_b': -1.1, 'null_t_mean': -0.01, 'null_t_sd': 1.41, 'null_fire': 1, 'null_seeds': 20}

## The hedge — EMLC is half dollar-basket FX

Regress EMLC excess on the UUP overlay excess: the slope is the FX exposure; the variance-min hedge ratio `b` is that slope (negative ⇒ a **long-UUP** overlay).

In [2]:
print(f"EMLC = a + b*UUP :  beta {R['emlc_uup_beta']:+.2f} (HAC t {R['t_emlc_uup']:+.2f})  R2 {R['emlc_uup_r2']:.2f}")
print(f"hedge ratio b   :  {R['hedge_b']:+.3f}  -> a long dollar-index overlay of |b| x NAV")

EMLC = a + b*UUP :  beta -1.12 (HAC t -11.91)  R2 0.50
hedge ratio b   :  -1.118  -> a long dollar-index overlay of |b| x NAV


## The race — excess-vs-excess (minus BIL cash)

In [3]:
print(f"unhedged EMLC : {R['unhedged_exc']:+.2f}%/yr  Sharpe {R['unhedged_sharpe']:+.2f}  (HAC t {R['t_unhedged']:+.2f})")
print(f"hedged   EMLC : {R['hedged_exc']:+.2f}%/yr  Sharpe {R['hedged_sharpe']:+.2f}  (HAC t {R['t_hedged']:+.2f})")
print(f"EMB (USD-EM)  : {R['emb_exc']:+.2f}%/yr  Sharpe {R['emb_sharpe']:+.2f}  (HAC t {R['t_emb']:+.2f})")
print(f"hedged - EMB  : {R['prem_diff']:+.2f}%/yr  (HAC t {R['t_prem_diff']:+.2f}, Welch {R['welch']:+.2f})  <- NEGATIVE")
print(f"confirms: LEMB Sharpe {R['lemb_sharpe']:+.2f} (t {R['lemb_t']:+.2f}), EBND {R['ebnd_sharpe']:+.2f} (t {R['ebnd_t']:+.2f})")

unhedged EMLC : +0.33%/yr  Sharpe +0.03  (HAC t +0.14)
hedged   EMLC : +1.62%/yr  Sharpe +0.20  (HAC t +0.94)
EMB (USD-EM)  : +3.10%/yr  Sharpe +0.34  (HAC t +1.48)
hedged - EMB  : -1.48%/yr  (HAC t -0.81, Welch -0.49)  <- NEGATIVE
confirms: LEMB Sharpe +0.23 (t +0.99), EBND +0.26 (t +1.16)


## Bootstrap Sharpe CIs (circular block) — does the carry clear zero?

In [4]:
print(f"unhedged EMLC : Sharpe {R['unhedged_sharpe']:+.2f}  95% CI [{R['boot_unh_lo']:+.2f}, {R['boot_unh_hi']:+.2f}]")
print(f"hedged   EMLC : Sharpe {R['hedged_sharpe']:+.2f}  95% CI [{R['boot_hed_lo']:+.2f}, {R['boot_hed_hi']:+.2f}]  frac<0 {R['boot_hed_fracneg']:.2f}  <- straddles 0")
print(f"EMB (USD-EM)  : Sharpe {R['emb_sharpe']:+.2f}  95% CI [{R['boot_emb_lo']:+.2f}, {R['boot_emb_hi']:+.2f}]")

unhedged EMLC : Sharpe +0.03  95% CI [-0.38, +0.47]
hedged   EMLC : Sharpe +0.20  95% CI [-0.21, +0.69]  frac<0 0.17  <- straddles 0
EMB (USD-EM)  : Sharpe +0.34  95% CI [-0.05, +0.91]


## Walk-forward hedge (36m rolling b, lag 1) — no look-ahead

The in-sample `b` is an FX-strip upper bound; the implementable rolling hedge gives the same thin result and shows the residual EM-FX the DXY proxy can't reach.

In [5]:
print(f"walk-forward hedged: {R['wf_exc']:+.2f}%/yr  Sharpe {R['wf_sharpe']:+.2f}  (HAC t {R['wf_t']:+.2f})")
print(f"residual EM-FX beta left by the proxy: {R['wf_resid_fx']:+.2f}  (the DXY basket != the EM basket)")

walk-forward hedged: +1.60%/yr  Sharpe +0.20  (HAC t +0.79)
residual EM-FX beta left by the proxy: +0.10  (the DXY basket != the EM basket)


## Robustness — two eras (split 2021-01-01)

In [6]:
print(f"2010-2020 (n={R['era_early_n']}): hedged {R['era_early_hed']:+.2f}%/yr (t {R['era_early_t']:+.2f})  prem-vs-EMB {R['era_early_prem']:+.2f}% (t {R['era_early_prem_t']:+.2f})")
print(f"2021-2026 (n={R['era_late_n']}): hedged {R['era_late_hed']:+.2f}%/yr (t {R['era_late_t']:+.2f})  prem-vs-EMB {R['era_late_prem']:+.2f}% (t {R['era_late_prem_t']:+.2f})")

2010-2020 (n=125): hedged +1.76%/yr (t +0.70)  prem-vs-EMB -3.58% (t -1.87)
2021-2026 (n=66): hedged +1.34%/yr (t +0.84)  prem-vs-EMB +2.50% (t +0.71)


## The timer — cost the overlay

In [7]:
print(f"overlay charge {R['cost_charge']:.2f}%/yr:  gross {R['cost_gross']:+.2f} -> net {R['cost_net']:+.2f}%/yr (t {R['cost_net_t']:+.2f}, Sharpe {R['cost_net_sharpe']:+.2f})")
print(f"net premium vs EMB: {R['cost_net_prem']:+.2f}%/yr (t {R['cost_net_prem_t']:+.2f})  <- the USD-EM ETF dominates")

overlay charge 0.46%/yr:  gross +1.62 -> net +1.15%/yr (t +0.67, Sharpe +0.15)
net premium vs EMB: -1.95%/yr (t -1.07)  <- the USD-EM ETF dominates


## Synthetic positive control — the machinery is unbiased

Live: recover a *planted* local carry, stay silent on the null. No network.

In [8]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from em_hedged import data, strategy as st
planted = st.synthetic_detect(data.synthetic_world(carry_annual=0.04, seed=906))
null_t = np.array([st.synthetic_detect(data.synthetic_world(carry_annual=0.0, seed=906+s))['t_hedged'] for s in range(20)])
print(f"planted (carry=4%/yr): hedged {planted['hedged_exc_ann_pct']:+.2f}%/yr, HAC t {planted['t_hedged']:+.2f}")
print(f"null (carry=0), 20 seeds: HAC t mean {null_t.mean():+.2f} (sd {null_t.std(ddof=1):.2f}), |t|>=2 in {(abs(null_t)>=2).sum()}/20")

planted (carry=4%/yr): hedged +5.53%/yr, HAC t +3.25
null (carry=0), 20 seeds: HAC t mean -0.01 (sd 1.41), |t|>=2 in 1/20


## Verdict

- **Signal — WEAK.** The FX-strip is a **real mechanism** — EMLC is ~50 % dollar-basket FX (β = -1.12, HAC *t* = -11.91), hedging lifts the excess-of-cash Sharpe +0.03 → +0.20 and halves the drawdown (-32% → -17%), the same on LEMB/EBND. **But the residual local carry is not robust**: +1.62 %/yr at HAC *t* = +0.94, a bootstrap Sharpe CI [-0.21, +0.69] straddling zero, and it **loses to USD-EM debt** (hedged − EMB = -1.48 %/yr). The 20-seed synthetic control recovers a planted carry (*t* = +3.25) and fires on 1/20 nulls, so the thin result is a true small edge, not a biased estimator. Short ~15-year sample, one dollar super-cycle.
- **Tradability — MIRAGE.** Even gross the hedged carry (Sharpe +0.20) is dominated by plain EMB (Sharpe +0.34); after the 0.46 %/yr overlay cost net is +1.15 %/yr (*t* +0.67) and the net premium vs EMB is -1.95 %/yr. A cheaper, simpler ETF wins — the local-hedged edge is a mirage.